# MENTORA — Train M4: Career-Skill Semantic Matcher (Sentence-Transformers)

Target: **Precision@5 >= 0.75**.

**Schema note:** our `training_pairs.csv` has columns `(required_skills, job_title, label)` where `required_skills` is a stringified Python list, not the doc's `(skill_text, career_text, label)` — the load cell below adapts for this.

## 0. Shared setup — mount Drive, confirm GPU, install deps
Run these three cells first in every notebook (per Phase 4 §0).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
# Expect to see "Tesla T4" — if this errors or shows no GPU, go to
# Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU -> Save, then re-run.

In [ ]:
import os
DATA = '/content/drive/MyDrive/mentora_data'
MODELS = '/content/drive/MyDrive/mentora_models'
for m in ['model1_gap_classifier', 'model2_question_generator',
          'model3_trajectory_predictor', 'model4_career_matcher', 'model5_cv_ner']:
    os.makedirs(f'{MODELS}/{m}', exist_ok=True)

# NOTE: upload the repo's datasets/ folder to Google Drive at this path first
# (mentora_data/{raw,processed,labeled}/...) — see datasets/README.md.

In [ ]:
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121  # Colab GPU wheel — do NOT reuse the CPU wheel from the backend's laptop plan
!pip install -q transformers peft accelerate sentence-transformers datasets evaluate seqeval spacy sacrebleu rouge_score

### Optional — Weights & Biases logging
Nice for live loss curves / a thesis screenshot. Skip entirely if you'd rather keep things simple — nothing below strictly needs it.

In [ ]:
# !pip install -q wandb
# import wandb
# wandb.login()  # paste your free-tier API key when prompted, once per session

## 1. Load base model + training pairs

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
import pandas as pd, ast

model = SentenceTransformer('all-mpnet-base-v2')  # downloads once, ~420MB
pairs = pd.read_csv(f'{DATA}/processed/m4/training_pairs.csv')
pairs['required_skills'] = pairs['required_skills'].apply(ast.literal_eval)
pairs['skill_text'] = pairs['required_skills'].apply(lambda skills: ', '.join(skills))
pairs = pairs.rename(columns={'job_title': 'career_text'})

# held-out split at the ROW level here is fine since each row is one
# (career, shuffled-or-own title) pair, not a repeated student -- no
# near-duplicate leak risk like Phase 4 §8 warns about for other models
train_pairs = pairs.sample(frac=0.85, random_state=42)
val_pairs = pairs.drop(train_pairs.index)

train_examples = [
    InputExample(texts=[row['skill_text'], row['career_text']], label=float(row['label']))
    for _, row in train_pairs.iterrows()
]
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)
train_loss = losses.CosineSimilarityLoss(model)

## 2. Precision@5 evaluator — the real target metric

In [ ]:
from sentence_transformers.evaluation import SentenceEvaluator
import numpy as np

class PrecisionAtKEvaluator(SentenceEvaluator):
    def __init__(self, skill_texts, career_texts, relevant_career_indices, k=5):
        self.skill_texts = skill_texts
        self.career_texts = career_texts
        self.relevant = relevant_career_indices
        self.k = k

    def __call__(self, model, output_path=None, epoch=-1, steps=-1):
        skill_emb = model.encode(self.skill_texts, convert_to_numpy=True)
        career_emb = model.encode(self.career_texts, convert_to_numpy=True)
        precisions = []
        for i, s_emb in enumerate(skill_emb):
            sims = career_emb @ s_emb / (np.linalg.norm(career_emb, axis=1) * np.linalg.norm(s_emb) + 1e-9)
            top_k = np.argsort(-sims)[:self.k]
            hits = len(set(top_k) & self.relevant[i])
            precisions.append(hits / self.k)
        score = float(np.mean(precisions))
        print(f'[epoch {epoch}] Precision@{self.k} = {score:.3f}')
        return score

career_profiles = pd.read_csv(f'{DATA}/processed/m4/career_profiles.csv')
career_profiles['required_skills'] = career_profiles['required_skills'].apply(ast.literal_eval)
all_career_texts = career_profiles['title'].tolist()
title_to_idx = {t: i for i, t in enumerate(all_career_texts)}

val_positive = val_pairs[val_pairs['label'] == 1.0]
val_skill_texts = val_positive['skill_text'].tolist()
val_relevant_indices = [{title_to_idx[t]} for t in val_positive['career_text']]

## 3. Fine-tune

In [ ]:
evaluator = PrecisionAtKEvaluator(val_skill_texts, all_career_texts, val_relevant_indices, k=5)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=4,
    evaluator=evaluator,
    evaluation_steps=200,
    output_path=f'{MODELS}/model4_career_matcher',
    save_best_model=True,
)

## Target metric: Precision@5 >= 0.75

If it plateaus lower:
- Add hard negatives (similar-sounding but non-matching career pairs) to `training_pairs.csv`
- Try `epochs=8`
- Consider swapping the starter `career_profiles.csv` for a real O*NET join (see datasets/SOURCES.md) — broader, more realistic skill vocabulary tends to help this metric directly